# 🎙️ ECE22073 Podcast Summarizer — Colab Runtime

**Repo:** [github.com/victoras136/asr-notebook](https://github.com/victoras136/asr-notebook)

This notebook runs the ASR pipeline and podcast generation on Colab GPU.
Your local Streamlit app communicates via Google Drive Bridge.

---
**⚠️ IMPORTANT:** Run each cell in order (Ctrl+Enter). Do not skip cells.

**Hardware:** Runtime → Change runtime type → **T4 GPU**

In [20]:
import warnings
warnings.filterwarnings("ignore")

import logging
logging.basicConfig(level=logging.INFO,
  format="%(asctime)s | %(levelname)s | %(message)s", force=True)

# Cell 1: Clone or pull the repo (private — reads token from Colab Secret)
import os
from google.colab import userdata
os.environ["OPENAI_API_KEY"] = userdata.get("OPENAI_API_KEY")
import subprocess, sys, os
from pathlib import Path

os.chdir("/content")
REPO_DIR = Path("/content/asr-notebook")

# Remove any deeply-nested clone from running cell 1 in wrong CWD
!rm -rf /content/asr-notebook/asr-notebook 2>/dev/null

from google.colab import userdata
TOKEN = userdata.get("GITHUB_TOKEN")
REPO_URL = f"https://{TOKEN}@github.com/victoras136/asr-notebook"

if not REPO_DIR.exists():
    print("Cloning private repo...")
    subprocess.run(["git", "clone", REPO_URL, str(REPO_DIR)], check=True)
else:
    print("Pulling latest...")
    subprocess.run(["git", "-C", str(REPO_DIR), "remote", "set-url", "origin", REPO_URL], check=True)
    subprocess.run(["git", "-C", str(REPO_DIR), "pull", "origin", "main"], check=True)

if not REPO_DIR.exists():
    raise RuntimeError("Repo not found after clone! Check token has repo scope.")

# Clear stale bytecode
!find /content/asr-notebook -name "__pycache__" -type d -exec rm -rf {} + 2>/dev/null

sys.path.insert(0, str(REPO_DIR / "Pipeline"))
print(f"Repo ready at {REPO_DIR.resolve()}")

Pulling latest...
Repo ready at /content/asr-notebook


In [ ]:
import warnings
warnings.filterwarnings("ignore")

# Cell 2: Install system deps + Python packages (5-10 min)
!apt-get install -y ffmpeg espeak-ng > /dev/null 2>&1
!pip install -q -r /content/asr-notebook/App/requirements_colab.txt 2>&1 | tail -5
!pip install --upgrade -q torchvision 2>&1 | tail -1
!pip install -q git+https://github.com/nari-labs/dia.git 2>&1 | tail -3
!pip install -q git+https://github.com/suno-ai/bark.git 2>&1 | tail -3
!pip install -q git+https://github.com/coqui-ai/TTS.git 2>&1 | tail -3
!pip install -q git+https://github.com/SWivid/F5-TTS.git 2>&1 | tail -3
print("All dependencies installed.")

In [ ]:
import warnings
warnings.filterwarnings("ignore")

# Cell 3: Authenticate HuggingFace (required for pyannote.audio + TTS models)
from huggingface_hub import notebook_login
notebook_login()

In [ ]:
import logging
logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s | %(levelname)s | %(message)s",
    force=True
)

import os, sys, time
from google.colab import userdata
os.environ["OPENAI_API_KEY"] = userdata.get("OPENAI_API_KEY")
os.environ["HF_TOKEN"] = userdata.get("HF_TOKEN")

from google.colab import drive
drive.mount('/content/drive')

sys.path.insert(0, '/content/asr-notebook/Pipeline')
os.chdir('/content/asr-notebook')

import config
import drive_bridge as db
import colab_job_watcher as cjw

db.init_drive_structure()
processed_names = set()
print("Watcher starting (Drive API mode) — Runtime → Interrupt execution to stop")

while True:
    try:
        for file_info in db.find_new_input_files():
            fname = file_info["name"]
            if not (fname.lower().endswith(".wav") or fname.lower().endswith(".mp3") or fname.lower().endswith(".m4a")):
                continue
            if fname in processed_names:
                continue
            processed_names.add(fname)
            cjw._handle_asr_job(file_info)

        for file_info in db.find_new_podcast_jobs():
            fname = file_info["name"]
            if fname in processed_names:
                continue
            processed_names.add(fname)
            cjw._handle_podcast_job(file_info)
    except Exception as e:
        logging.error("Watcher error: %s", e, exc_info=True)
    time.sleep(config.POLL_INTERVAL_SEC)


Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Watcher starting — press Runtime → Interrupt execution to stop


Using cache found in /root/.cache/torch/hub/snakers4_silero-vad_master
ERROR:llm_integration:LLM call failed: Error code: 401 - {'error': {'message': 'Incorrect API key provided: ollama. You can find your API key at https://platform.openai.com/account/api-keys.', 'type': 'invalid_request_error', 'param': None, 'code': 'invalid_api_key'}}
ERROR:llm_integration:LLM call failed: Error code: 401 - {'error': {'message': 'Incorrect API key provided: ollama. You can find your API key at https://platform.openai.com/account/api-keys.', 'type': 'invalid_request_error', 'param': None, 'code': 'invalid_api_key'}}
ERROR:summary_generator:LLM call failed: Error code: 401 - {'error': {'message': 'Incorrect API key provided: ollama. You can find your API key at https://platform.openai.com/account/api-keys.', 'type': 'invalid_request_error', 'param': None, 'code': 'invalid_api_key'}}
ERROR:summary_generator:LLM call failed: Error code: 401 - {'error': {'message': 'Incorrect API key provided: ollama. Yo

---
## Monitoring

- **Drive output:** Check `MyDrive/ece22073/output/{job_id}/status.json` for live progress
- **Local Streamlit:** Polls `status.json` every 5 seconds
- **Stall detection:** If status hasn't updated in 10 minutes → "stalled" warning in Streamlit
- **Resume:** If Colab disconnects, just re-run cells 1-4 — input files remain untouched

In [ ]:
import logging
logging.basicConfig(level=logging.INFO,
  format="%(asctime)s | %(levelname)s | %(message)s", force=True)

import warnings
warnings.filterwarnings("ignore")

import json, os, sys
from pathlib import Path

sys.path.insert(0, '/content/asr-notebook/Benchmarks')
sys.path.insert(0, '/content/asr-notebook/Pipeline')

try:
    import jiwer, rouge_score
    print("jiwer + rouge_score OK")
except ImportError:
    print("Missing: pip install jiwer rouge-score")
    print("Skipping WER evaluation.")
else:
    from evaluate_real_pipeline import run_real_evaluation
    success = run_real_evaluation()
    print(f"\nEvaluation {'PASSED' if success else 'FAILED'}")

    # Side-by-side: GT vs normalized output
    results_dir = Path('/content/asr-notebook/Results')
    gt_path = results_dir / "ground_truth.json"
    norm_path = results_dir / "normalized_transcript.txt"

    if norm_path.exists():
        norm_text = norm_path.read_text(encoding='utf-8')
        gt_text = ""
        if gt_path.exists():
            with open(gt_path) as f:
                gt_text = json.load(f).get("transcript", "")
        print("\n=== Ground Truth (first 300 chars) ===")
        print(gt_text[:300] or "(no GT file)")
        print("\n=== Normalized Pipeline Output (first 300 chars) ===")
        print(norm_text[:300])
    else:
        print("No normalized_transcript.txt found — run the pipeline first.")